In [1]:
# MOUNT DRIVE FIRST
from google.colab import drive
drive.mount('/content/drive')

# CHECK RESUME STATUS
import os
import json

STATE_FILE = "training_state.json"
CHECKPOINT_DIR = "/content/drive/MyDrive/codellama-coding-chat/checkpoint-250"

print("🔍 Checking resume status...")

# Load state (optional)
if os.path.exists(STATE_FILE):
    with open(STATE_FILE) as f:
        state = json.load(f)
    print(f"✅ State: Step {state.get('last_step', 'unknown')}")
else:
    print("⚠️  No state file (optional)")
    state = None

# Check Drive checkpoint
if os.path.exists(CHECKPOINT_DIR):
    print(f"✅ Checkpoint: {CHECKPOINT_DIR}")
    files = os.listdir(CHECKPOINT_DIR)
    print(f"   Files: {len(files)} items")
    RESUME_CHECKPOINT = CHECKPOINT_DIR
else:
    print(f"❌ Checkpoint not found: {CHECKPOINT_DIR}")
    raise SystemExit

print("\n🚀 READY TO RESUME!")

Mounted at /content/drive
🔍 Checking resume status...
⚠️  No state file (optional)
✅ Checkpoint: /content/drive/MyDrive/codellama-coding-chat/checkpoint-250
   Files: 12 items

🚀 READY TO RESUME!


In [2]:
# COPY FROM DRIVE TO LOCAL (faster training)
import shutil

local_dir = "./codellama-coding-chat/checkpoint-250"
os.makedirs("./codellama-coding-chat", exist_ok=True)

print(f"📥 Copying checkpoint to local...")
print(f"   From: {RESUME_CHECKPOINT}")
print(f"   To: {local_dir}")

if os.path.exists(local_dir):
    print("   Local already exists, skipping copy")
else:
    shutil.copytree(RESUME_CHECKPOINT, local_dir)
    print("   ✅ Copied!")

RESUME_CHECKPOINT = local_dir
print(f"✅ Using: {RESUME_CHECKPOINT}")

📥 Copying checkpoint to local...
   From: /content/drive/MyDrive/codellama-coding-chat/checkpoint-250
   To: ./codellama-coding-chat/checkpoint-250
   ✅ Copied!
✅ Using: ./codellama-coding-chat/checkpoint-250


In [3]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 39.6 MB/s eta 0:00:00


In [4]:
import torch
import os
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig,
    TrainerCallback
)
from peft import PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset, concatenate_datasets

print(f"✅ CUDA: {torch.cuda.is_available()}")
print(f"✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

✅ CUDA: True
✅ GPU: Tesla T4


In [5]:
# CONFIG
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
OUTPUT_DIR = "./codellama-coding-chat"
RESUME_CHECKPOINT = "./codellama-coding-chat/checkpoint-250"

MAX_SEQ_LENGTH = 512
CODE_SAMPLES = 3000
CHAT_SAMPLES = 1000

# NEW: Train to 500 this time
MAX_STEPS = 500
SAVE_STEPS = 100

print(f"📁 Output: {OUTPUT_DIR}")
print(f"⏮️  Resuming from: {RESUME_CHECKPOINT}")
print(f"🎯 Target: Step {MAX_STEPS}")

📁 Output: ./codellama-coding-chat
⏮️  Resuming from: ./codellama-coding-chat/checkpoint-250
🎯 Target: Step 500


In [6]:
print("\n📚 Loading datasets...")

code_dataset = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split="train")
code_dataset = code_dataset.shuffle(seed=42).select(range(min(CODE_SAMPLES, len(code_dataset))))
print(f"✅ Code: {len(code_dataset)}")

chat_dataset = load_dataset("mlabonne/orpo-dpo-mix-40k", split="train")
chat_dataset = chat_dataset.shuffle(seed=42).select(range(min(CHAT_SAMPLES, len(chat_dataset))))
print(f"✅ Chat: {len(chat_dataset)}")


📚 Loading datasets...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/905 [00:00<?, ?B/s]

data/train-00000-of-00001-8b6e212f3e1ece(…):   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18612 [00:00<?, ? examples/s]

✅ Code: 3000


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/127M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/44245 [00:00<?, ? examples/s]

✅ Chat: 1000


In [7]:
def format_code(example):
    instruction = example.get('prompt', example.get('instruction', ''))
    input_text = example.get('input', '')
    output = example.get('completion', example.get('output', ''))
    prompt = f"{instruction}\n\nInput: {input_text}" if input_text else instruction
    return f"[INST] {prompt.strip()} [/INST] {output.strip()}"

def format_chat(example):
    prompt = example.get('prompt', example.get('question', example.get('instruction', '')))
    response = example.get('chosen', example.get('response', example.get('answer', example.get('output', ''))))
    if isinstance(response, list):
        response = response[0] if response else ""
    return f"[INST] {prompt.strip()} [/INST] {str(response).strip()}"

print("Formatting...")
code_formatted = code_dataset.map(lambda x: {"text": format_code(x)}, remove_columns=code_dataset.column_names)
chat_formatted = chat_dataset.map(lambda x: {"text": format_chat(x)}, remove_columns=chat_dataset.column_names)

full_dataset = concatenate_datasets([code_formatted, chat_formatted]).shuffle(seed=42)
print(f"✅ Total: {len(full_dataset)}")

Formatting...


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

✅ Total: 4000


In [23]:
print("\n🚀 Loading model from checkpoint...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model)

# Load your adapter from checkpoint
model = PeftModel.from_pretrained(base_model, RESUME_CHECKPOINT)

print(f"✅ Loaded from: {RESUME_CHECKPOINT}")
print(f"💾 GPU: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


🚀 Loading model from checkpoint...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Loaded from: ./codellama-coding-chat/checkpoint-250
💾 GPU: 9.48 GB


In [36]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,       # ← REDUCED: 1 instead of 2
    gradient_accumulation_steps=8,       # ← INCREASED: 8 instead of 4 (same effective batch)
    optim="paged_adamw_8bit",            # ← CHANGED: back to paged_adamw_8bit (valid and compatible with fp16)
    learning_rate=2e-4,
    max_grad_norm=0.3,
    warmup_steps=15,                     # ← CHANGED: from 0.03 (float) to 15 (int) based on 0.03 * 500 max_steps
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant':False},
    report_to="none",
    max_steps=500,
)

In [37]:
class ProgressCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        with open(os.path.join(args.output_dir, "latest_checkpoint.txt"), "w") as f:
            f.write(f"{state.global_step}")
        print(f"💾 Checkpoint: Step {state.global_step}")

print("✅ Callback ready")

✅ Callback ready


In [38]:
print("\n🔧 Setting up trainer...")

from transformers import Trainer, DataCollatorForLanguageModeling

# Tokenize dataset first
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length"
    )

print("Tokenizing dataset...")
tokenized_dataset = full_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=full_dataset.column_names
)

# Data collator for causal LM
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # We're doing autoregressive, not masked
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    callbacks=[ProgressCallback],
)

print("✅ Trainer ready!")
print(f"📊 Dataset: {len(tokenized_dataset)} examples")
print(f"🎯 Steps: 250 → {MAX_STEPS}")


🔧 Setting up trainer...
Tokenizing dataset...
✅ Trainer ready!
📊 Dataset: 4000 examples
🎯 Steps: 250 → 500


In [39]:
print("\n🔥 RESUMING TRAINING!")
print(f"💻 GPU: {torch.cuda.get_device_name(0)}")  # Auto-detect
print(f"🎯 250 → 500")
print("=" * 50)

trainer.train()

print("=" * 50)
print("✅ Training complete!")


🔥 RESUMING TRAINING!
💻 GPU: Tesla T4
🎯 250 → 500


AssertionError: No inf checks were recorded for this optimizer.